In [227]:
import pandas as pd
import ollama
model = "llama4:latest"
#model = "llama3.2:latest"
#model = "llama3.1:8b"

In [246]:
# Define the inclusion criteria
prompt = """ 
IMPORTANT: 
Your response MUST be a single line, comma-separated, in this exact order:
  classification, species, country, statistics, optimization, strategies, notes
  Example:
  true, pigs, germany, deterministic modelling, optimization, household surveillance, meets inclusion criteria
  false, birds, brazil, systematic review, unknown, unknown, vaccination, excluded due to being a review

  Do NOT use ";", newlines, or any other format.
  Do not use any additional text or explanations outside the specified 7 colums format.

  INSTRUCTIONS:
You are a systematic review expert, evaluate every provided article title {Title} and abstract {Abstract} against the specified inclusion criteria.

1. Carefully read the article title and abstract.
   Exclude any literature review articles, meta-analyses, or laboratory tecnhiques, cell culture, PCR or ELISA techniques that do not focus on specific animal species or diseases.
2. Assess whether the article meets ALL of the following inclusion criteria:

   a. Modelling: "The study must involve applied statistical modelling, mathematical simulations, or predictive analytics for disease surveillance."
   b. Optimization: "The study must actively apply optimization methods (e.g., reinforcement learning, algorithmic enhancements or others) to improve efficiency, decision-making, or predictive accuracy in disease surveillance."
   
   
3. Provide your assessment using ONLY one of these two responses. 
  - If ALL criteria are met: true
  - If ANY criterion is not met: false
  Explain on notes maximum 10 words the logic of the true/false classification. Not additional text only the corresponding field. Not additional fields.

4. Extract the following information:
  - Species: List of studied animal species using common names. If only latin names are provided and no common name exists, retain Latin names. Do not classify disease as specie.
  - Country: Specify the country where the study was conducted. (If multiple countries are involved, list them all. If unclear, write 'unknown')
  - Statistics: List the statistical methods applied in the study (e.g. regression analysis, probabilistic methods, mechanistics models, bayesian inference, compartment models, or machine learning). If unclear, write 'unknown')
  - Optimization: "Indicate whether optimization methods (e.g., reinforcement learning, algorithmic enhancements or others) were actively applied. If optimization is only mentioned but not implemented, write 'mentioned but not applied' instead of 'unknown'.
  - Strategies: Extract the recomended surveillance strategies (e.g., risk-based monitoring, early-warning models). If strategies are unclear or not specified, write 'unknown')
  - All values must be lowercase. 
  - Multiples species/countries/statistics/strategies should be separated by spaces.

5 . Follow these strict rules:
- Base your assessment solely on the information provided in the title and abstract.
- If any information is unclear or not explicitly stated, write 'unknown'.
- Ensure your response is a single line, comma-separated, in lowercase, without any additional characters or brake lines \n.
- If multiple species/countries/statistics/strategies are mentioned, include them all in the corresponding field.
- If optimization is mentioned but not actively applied, write 'mentioned but not applied'. For all other unclear information, write 'unknown.
- ONLY use one line DO NOT deviate from the format.

Your task is to provide a clear, binary assessment and extract the information as described above, single line format."""

In [247]:
def classify_row(row):
    prompt_text = f"{prompt}\n\nTitle: {row['Title']}\nAbstract: {row['Abstract']}"
    response = ollama.generate(
        model=model,
        prompt=prompt_text
    )
    # Expecting output like: true, dog, usa, logistic regression, 
    parts = [p.strip() for p in response['response'].strip().lower().split(',')]
    # Ensure we always have 7 parts
    while len(parts) < 7:
        parts.append('unknown')
    classification, species, country, statistics, optimization, strategies, notes = parts[:7]
    return pd.Series({
        'classification': 'included' if classification == 'true' else 'not included',
        'species': species,
        'country': country,
        'statistics': statistics,
        'optimization': optimization,
        'strategies': strategies,
        'notes': notes
    })


In [249]:

# Load the CSV file
df = pd.read_csv('pubmed_articles_asf.csv', delimiter=',')

# Apply the classification function to each row and expand the results into new columns
df[['classification', 'species', 'country', 'statistics', 'optimization', 'strategies', 'notes']] = df.apply(classify_row, axis=1)

# Save the updated dataframe
df.to_csv('classified_papers_asf.csv', index=False)

In [250]:
def classify_row(row):
    prompt_text = f"{prompt}\n\nTitle: {row['Title']}\nAbstract: {row['Abstract']}"
    response = ollama.generate(
        model=model,
        prompt=prompt_text
    )
    # Expecting output like: true, dog, usa, logistic regression, ..., notes
    parts = [p.strip() for p in response['response'].strip().lower().split(',')]
    while len(parts) < 7:
        parts.append('unknown')
    classification, species, country, statistics, optimization, strategies, notes = parts[:7]
    return pd.Series({
        'classification': 'included' if classification == 'true' else 'not included',
        'species': species,
        'country': country,
        'statistics': statistics,
        'optimization': optimization,
        'strategies': strategies,
        'notes': notes
    })

In [251]:
# Print statistics about the classification
total = len(df)
included = (df['classification'] == 'included').sum()
not_included = (df['classification'] == 'not included').sum()

print(f"Total articles classified: {total}")
print(f"Included: {included}")
print(f"Not included: {not_included}")

# Filter only the included articles
idf = df[df['classification'] == 'included']


# Print a table with the number of articles per country
print("\nNumber of articles per country:")
print(idf['country'].value_counts().to_frame('count'))

# Print a table with the optimization strategies used   
print("\nOptimization:")
print(idf['optimization'].value_counts().to_frame('count'))

# Print a table with the statistical analysisused   
print("\nstatistics:")
print(idf['statistics'].value_counts().to_frame('count'))

# Print a table with the strategies   
print("\nstrategies:")
print(idf['strategies'].value_counts().to_frame('count'))

# Print a table with the species   
print("\nspecies:")
print(idf['species'].value_counts().to_frame('count'))

Total articles classified: 38
Included: 11
Not included: 27

Number of articles per country:
                    count
country                  
poland                  3
united states           1
unknown                 1
sus scrofa              1
russian federation      1
vietnam                 1
lithuania               1
germany                 1
usa                     1

Optimization:
                     count
optimization              
optimization            10
capreolus capreolus      1

statistics:
                                               count
statistics                                          
mechanistic modelling                              2
deterministic modelling                            1
spatially explicit disease transmission model      1
stochastic modelling                               1
spatially explicit transmission models             1
roe deer                                           1
logistic regression                                1
simulati

In [275]:
print(human_df.columns.tolist())

['PMID', 'First Author', 'Publication Date', 'Title', 'Abstract', 'DOI', 'Human']


In [279]:
# Print the first 10 included articles: only Title and classification response variables
print(idf[['Title', 'classification', 'statistics']].head(10))

                                                Title classification  \
1   Control of African swine fever epidemics in in...       included   
7   Ecological drivers of African swine fever viru...       included   
13  Optimising response to an introduction of Afri...       included   
15  A Mathematical Model that Simulates Control Op...       included   
17  Social structure defines spatial transmission ...       included   
19  Not Just Pictures: Utility of Camera Trapping ...       included   
21  Spatio-temporal modeling of the African swine ...       included   
24  Early-phase risk assessments during the first ...       included   
25  Optimizing Vaccination Strategies against Afri...       included   
31  First-Passage Time Analysis Based on GPS Data ...       included   

                                       statistics  
1                         deterministic modelling  
7                           mechanistic modelling  
13  spatially explicit disease transmission model  

In [ ]:
# Comparing to human classified articles


# Load the human_classified file (make sure to use the correct delimiter it changes in my german regional config)
human_df = pd.read_csv('human_classified.csv', delimiter=';')

print(human_df.columns.tolist())  # Check actual column names, be carefull

# some colum checking
df['PMID'] = df['PMID'].astype(str).str.strip()
human_df['PMID'] = human_df['PMID'].astype(str).str.strip()

# Use the correct column name below!
df['human'] = df['PMID'].isin(human_df.loc[human_df['Human'] == 1, 'PMID']).astype(int)

print(df['human'].sum())  # Should be 7
print(df[df['human'] == 1]['PMID'])  # Show  PMIDs onlz the ones that match (check on csv)

['PMID', 'First Author', 'Publication Date', 'Title', 'Abstract', 'DOI', 'Human']
7
1     27938676
13    35881004
17    33468025
21    25457602
22    29799177
31    40303815
33    33340089
Name: PMID, dtype: object


In [ ]:
# Now df has the new column
print(df[['PMID', 'human']].head())

       PMID  human
0  39937724      0
1  27938676      1
2  37306707      0
3  36700212      0
4  36680090      0


In [ ]:
#%pip install statsmodels
# Check the se and sp calc by hand

import numpy as np
from statsmodels.stats.proportion import proportion_confint

# Map classification to binary: 1 for 'included', 0 for 'not included'
df['pred'] = (df['classification'] == 'included').astype(int)
df['true'] = df['human']

# Confusion matrix components
tp = ((df['true'] == 1) & (df['pred'] == 1)).sum()
tn = ((df['true'] == 0) & (df['pred'] == 0)).sum()
fp = ((df['true'] == 0) & (df['pred'] == 1)).sum()
fn = ((df['true'] == 1) & (df['pred'] == 0)).sum()
total = len(df)

# Metrics
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
accuracy = (tp + tn) / total if total > 0 else np.nan

# 95% confidence intervals (using Wilson score interval)
sens_low, sens_upp = proportion_confint(tp, tp + fn, alpha=0.05, method='wilson')
spec_low, spec_upp = proportion_confint(tn, tn + fp, alpha=0.05, method='wilson')
acc_low, acc_upp = proportion_confint(tp + tn, total, alpha=0.05, method='wilson')

print(f"Sensitivity: {sensitivity:.3f} (95% CI: {sens_low:.3f}–{sens_upp:.3f})")
print(f"Specificity: {specificity:.3f} (95% CI: {spec_low:.3f}–{spec_upp:.3f})")
print(f"Accuracy:    {accuracy:.3f} (95% CI: {acc_low:.3f}–{acc_upp:.3f})")

# Save with metrics
df.to_csv('classified_papers_asf_metric.csv', index=False)

Sensitivity: 0.857 (95% CI: 0.487–0.974)
Specificity: 0.839 (95% CI: 0.674–0.929)
Accuracy:    0.842 (95% CI: 0.696–0.926)
